# Cagetti & De Nardi (2006) — Theory Reproduction with SolvingMicroDSOPs

**Paper:** Marco Cagetti and Mariacristina De Nardi, *"Entrepreneurship, Frictions, and Wealth,"* Journal of Political Economy, 114(5), October 2006, pp. 835–870.

**Source copy:** `Entrepreneurship, Frictions, and Wealth/Entrepreneurship, Frictions, and Wealth.tex` (in this folder).

## What this notebook does

This notebook is a **self-contained** reproduction of the **theory part** of the paper (sections II and III):

1. Re-states the Bellman system (eqs. 2–14) and the paper's parameter calibration    (Table 5, Appendix A).
2. **Maps** the paper's household problem onto the modular stage architecture    shipped with **SolvingMicroDSOPs** (`cons-noshocks`, `ModelParams`, `Stage`,    Endogenous Grid Method).
3. **Solves** the coupled value functions $V$, $V_w$, $V_e$, $W$, $W_e$, $W_r$    at the paper's **calibrated** structural parameters and a **fixed** price    environment ($r$, $w$, $\tau$, $p$).
4. **Inspects** the solution: value-function shapes, occupational branches,    entrepreneur $k^{\star}$, borrowing-enforcement incidence, comparative    statics.
5. **Approximates** the stationary distribution $\mu$ (a young-only kernel)    and compares the implied entrepreneur share to paper Table 6.

## What is *not* attempted here

- The **general-equilibrium** outer loop that clears the capital market and   the social-security budget (Appendix B of the paper).
- The **full demographic** stationary distribution with young $\leftrightarrow$   old flows and descendant-birth accounting.
- The **calibration** itself (Table 5's calibrated parameters are taken as   given).
- Table 6 / Table 7 moment matching beyond a qualitative comparison of the   entrepreneur share.

The boundaries are a consequence of working at **fixed prices**; see Section 11 for what would be needed to close the loop.

## How to run

Execute from the repository root so that `Code/Python` is on `sys.path` (the first code cell searches upward for it). The notebook takes roughly **90 seconds** end-to-end.


---
# Part I — Paper theory

This part summarises the dynamic program we will solve. Equation numbers follow the JPE publication. For a standalone paper-faithful reference (including the full Table 7 counterfactuals and Appendix B equilibrium closure), see `cagetti2006_dynamic_program_excerpt.md` in this folder.

## 1.1 Demographics and preferences

- Agents are either **young** or **old**.
- Young $\to$ old with probability $1-\pi_y$ each period; old survives with   probability $\pi_o$; old dies with probability $1-\pi_o$ and a   **descendant** is born.
- Period utility is CRRA: $u(c) = c^{1-\sigma}/(1-\sigma)$.
- Discount factor $\beta$; altruism weight $\eta$ on the descendant's value   (baseline $\eta=1$, perfect altruism).

## 1.2 State, controls, status

Full individual state $\mathbf{x}=(a, y, \theta, s)$ with status $s\in\{\text{YW}, \text{YE}, \text{OE}, \text{OR}\}$:

| Status | Meaning | State it carries |
|---|---|---|
| YW | young worker | $(a, y, \theta)$ |
| YE | young entrepreneur | $(a, y, \theta)$ |
| OE | old continuing entrepreneur | $(a, \theta)$ |
| OR | old retiree (absorbing) | $a$ |

Controls: $c$ (everyone), $a'$ (everyone), $k$ (entrepreneurs only). $k$ is **intra-period** working capital; $a'$ crosses periods.

## 1.3 Technology and budgets

- **Entrepreneurial technology:** $y^e = \theta k^{\nu}$ with $\nu<1$;   capital depreciates at rate $\delta$.
- **Corporate sector (GE only):** $Y_c = A K_c^{\alpha} L_c^{1-\alpha}$   pins down $(r, w)$ in equilibrium.
- **Worker budget:** $a' = (1+r)a + (1-\tau) w y - c$.
- **Retiree budget:** $a' = (1+r)a + p - c$.
- **Entrepreneur resource equation:**
  $$\underbrace{(1-\delta)k + \theta k^\nu}_{\text{output + undepreciated }k} - \underbrace{(1+r)(k-a)}_{\text{net borrowing cost}} - c \equiv a'.$$

## 1.4 Bellman system (eqs. 2–14)

Young choose occupation:
$$V(a,y,\theta) = \max\{V_w(a,y,\theta),\ V_e(a,y,\theta)\}. \tag{2}$$

**Young entrepreneur:**
$$V_e(a,y,\theta) = \max_{c,k,a'}\bigl\{u(c) + \beta\pi_y\,\mathbb{E}V(a',y',\theta') + \beta(1-\pi_y)\,\mathbb{E}W(a',\theta')\bigr\} \tag{3}$$
subject to
$$a' = (1-\delta)k + \theta k^\nu - (1+r)(k-a) - c \tag{4}$$
$$\underbrace{\text{value of honouring}}_{\text{LHS of eq. (3)}} \;\ge\; V_w(f\cdot k, y, \theta) \tag{5}$$
$$a\ge 0,\quad k\ge 0. \tag{6,7}$$

Equation (5) is the **enforcement constraint**: on default the entrepreneur absconds with a fraction $f$ of working capital and reverts to being a worker with assets $fk$. Higher $f$ $\Rightarrow$ default cheaper $\Rightarrow$ **tighter** endogenous borrowing limit.

**Young worker:**
$$V_w(a,y,\theta) = \max_{c,a'}\bigl\{u(c) + \beta\pi_y\,\mathbb{E}V(a',y',\theta') + \beta(1-\pi_y)\,W_r(a')\bigr\}. \tag{8,9}$$
A young worker who ages enters retirement **directly** ($W_r$, not $W$ — retirement is absorbing).

**Old:**
$$W(a,\theta) = \max\{W_e(a,\theta),\ W_r(a)\}. \tag{10}$$

**Old continuing entrepreneur:**
$$W_e(a,\theta) = \max_{c,k,a'}\bigl\{u(c) + \beta\pi_o\,\mathbb{E}W(a',\theta') + \eta\beta(1-\pi_o)\,\mathbb{E}V(a',y',\theta')\bigr\} \tag{11}$$
subject to (4), (6), (7), and
$$\text{LHS of (11)} \;\ge\; W_r(f\cdot k). \tag{12}$$

**Old retiree (absorbing):**
$$W_r(a) = \max_{c,a'}\bigl\{u(c) + \beta\pi_o W_r(a') + \eta\beta(1-\pi_o)\,\mathbb{E}V(a',y',\theta')\bigr\} \tag{13}$$
$$a' = (1+r)a + p - c. \tag{14}$$

## 1.5 Shock processes (Appendix A)

In the **baseline** calibration $y$ and $\theta$ are **independent** first-order Markov chains.

**Worker productivity** $y$ — $\log y$ is AR(1) with persistence 0.95, variance chosen to match the PSID earnings Gini of 0.38; Tauchen–Hussey discretisation to **five** states (mean normalised to 1):
$$y \in (0.2468,\ 0.4473,\ 0.7654,\ 1.3097,\ 2.3742),$$
with $5\times 5$ transition matrix $\mathbf{P}_y$ reproduced below.

**Entrepreneurial ability** $\theta\in(0, 0.514)$ with
$$\mathbf{P}_\theta = \begin{pmatrix}0.964 & 0.036\\ 0.206 & 0.794\end{pmatrix}.$$

## 1.6 Parameters (Table 5)

**Panel A — fixed from other studies:**

| Symbol | Value | Source |
|---|---|---|
| $\sigma$ | 1.5 | Attanasio et al. (1999) |
| $\delta$ | 0.06 | Stokey & Rebelo (1995) |
| $\alpha$ | 0.33 | Gollin (2002) |
| $A$ | 1.0 | normalisation |
| $\pi_y$ | 0.978 | text |
| $\pi_o$ | 0.911 | text |
| $\eta$ | 1.0 | perfect altruism (baseline) |
| $p$ | 40% of avg. yearly income | Kotlikoff et al. (1999) |
| $\mathbf{P}_y$ | Appendix A 5x5 | Storesletten et al. (2004) |

**Panel B — calibrated (joint moment-matching):**

| Symbol | Value | Meaning |
|---|---|---|
| $\beta$ | 0.865 | discount factor |
| $\theta$ grid | $(0, 0.514)$ | Table 5 rounds to 0.51 |
| $\mathbf{P}_\theta$ | see 1.5 | ability transition |
| $\nu$ | 0.88 | returns in $\theta k^\nu$ |
| $f$ | 0.75 | enforcement / absconding fraction |

## 1.7 Targets (Table 6 — baseline with entrepreneurs)

| Moment | Data | Model |
|---|---|---|
| Capital–output ratio $K/Y$ | 3.0 | 3.0 |
| Wealth Gini | 0.80 | 0.80 |
| Entrepreneur fraction | 7.55% | 7.50% |
| Top 1% wealth share | 30% | 31% |
| Top 5% | 54% | 60% |
| Top 20% | 81% | 83% |
| Top 40% | 94% | 94% |

Section 8 of this notebook approximates the entrepreneur fraction; the other moments require a full stationary distribution and GE closure.


---
# Part II — Mapping to SolvingMicroDSOPs

A paper-faithful standalone reference for the Bellman system, tables, and stage decomposition lives in `cagetti2006_dynamic_program_excerpt.md`. Part I above restates the equations; this part makes the **mapping to the SolvingMicroDSOPs toolkit** explicit.

## 2.1 SolvingMicroDSOPs pieces we use

- `ModelParams` (`Code/Python/solution.py`) — a lightweight container for   $\rho$ (CRRA), $R^f$, the asset grid, etc.
- `Stage` — a named perch-arvl-dcsn-cntn structure; each stage can expose   `v`, `c`, `v'`, `c_delta` on its decision perch.
- `solve_cons_noshocks` (`Code/Python/stages/cons_noshocks.py`) — solves the   shock-free consumption stage using the **Endogenous Grid Method** given a   continuation value function $\bar v(a')$ and consumed-function   $c_\delta(a') = (\bar v'(a'))^{-1/\sigma}$.
- `build_a_grid` — multi-exponential asset grid.

## 2.2 Stage decomposition of one period

Following the excerpt's §11, a single period is:

```
young_connector -> young_choice -> { young_worker | young_entrepreneur }
old_connector   -> old_choice   -> { old_entrepreneur | old_retiree }
```

| Stage | Kind | Paper eq. | How we implement it |
|---|---|---|---|
| `young_connector` | shock + aging | embedded in continuation | encoded in the weights $\beta\pi_y$ / $\beta(1-\pi_y)$ of $\bar v$ |
| `young_choice` | branch (discrete max) | (2) | $V = \max(V_w, V_e)$ pointwise on the grid |
| `young_worker` | cons-noshocks | (8)–(9) | `solve_cons_noshocks` with $m^w = (1+r)a + (1-\tau)wy$ |
| `young_entrepreneur` | optimise-$k$ $\to$ cons-noshocks | (3)–(7) | outer discrete search over $k$; inner `solve_cons_noshocks` on $m^e$; enforcement check vs $V_w(fk, y, \theta)$ |
| `old_connector` | shock + survival | embedded in continuation | weights $\beta\pi_o$ / $\eta\beta(1-\pi_o)$ |
| `old_choice` | branch (discrete max) | (10) | $W = \max(W_e, W_r)$ pointwise |
| `old_entrepreneur` | optimise-$k$ $\to$ cons-noshocks | (11)–(12) | same pipeline as young entrepreneur; enforcement vs $W_r(fk)$ |
| `old_retiree` | cons-noshocks | (13)–(14) | `solve_cons_noshocks` with $m^r = (1+r)a + p$ |

Three subtleties from the excerpt need to be respected in the implementation:

1. **Retirement is absorbing** (eq. 13). Once in OR there is no re-entry to    entrepreneurship — our `update_Wr` never references $W_e$.
2. **Young worker $\to$ old** goes to $W_r(a')$ **directly**, not to    $W = \max(W_e, W_r)$ (eq. 8). A young worker who ages enters retirement    — our `update_Vw` uses `interp_Wr` (not `interp_W`) for the aging term.
3. **Descendant inheritance** on an old agent's death: $y'$ is drawn from    the invariant distribution of $\mathbf{P}_y$, but $\theta'$ is    conditional on the deceased parent's $\theta$ (not on $\theta$'s    invariant). Our `EV_descendant` mixes `pi_y_stat[jy]` with    `P_theta[itheta_parent, jth]` accordingly.

## 2.3 Encoding the Bellman continuations

Every Bellman equation has the form $\max_{c,\ldots} u(c) + \bar v(a')$ where $\bar v$ is a **weighted sum of interpolated value functions**. For each sub-problem we build $\bar v(\cdot)$ as a closure and hand it to `solve_cons_noshocks`. The package's `DiscFac` is set to 1 to avoid double-counting — all paper weights ($\beta\pi_y$, $\beta(1-\pi_y)$, $\beta\pi_o$, $\eta\beta(1-\pi_o)$) appear **inside** $\bar v$:

| Bellman | Paper eq. | $\bar v(a')$ (this notebook) |
|---|---|---|
| $V_w$ | (8) | $\beta\pi_y\,\mathbb{E}V(a',y',\theta') + \beta(1-\pi_y)\,W_r(a')$ |
| $V_e$ | (3) | $\beta\pi_y\,\mathbb{E}V(a',y',\theta') + \beta(1-\pi_y)\,\mathbb{E}W(a',\theta')$ |
| $W_e$ | (11) | $\beta\pi_o\,\mathbb{E}W(a',\theta') + \eta\beta(1-\pi_o)\,\mathbb{E}V_{\text{desc}}(a',y',\theta')$ |
| $W_r$ | (13) | $\beta\pi_o\,W_r(a') + \eta\beta(1-\pi_o)\,\mathbb{E}V(a',y',\theta')$ |

## 2.4 The entrepreneur's $k$ choice and the enforcement constraint

`cons-noshocks` only handles the $c$-subproblem given cash-on-hand. For the entrepreneur we therefore solve the $c$-stage **once per $(y, \theta)$** to obtain the interpolated $v(m^e)$ and $c(m^e)$, then do a **discrete outer search over $k$** at each $a$. At each $(a, y, \theta, k)$:

- Compute $m^e(a,k,\theta) = (1-\delta)k + \theta k^\nu - (1+r)(k-a)$.
- Evaluate $v_{\text{honour}} = v(m^e)$ from the EGM stage.
- Compare to $v_{\text{default}} = V_w(fk, y, \theta)$ (young) or   $W_r(fk)$ (old).
- Keep $k$ only if eq. (5) / (12) holds; pick the argmax.

This is equivalent to the standard reformulation of the paper's implicit borrowing limit as a Bellman-level participation constraint — the limit is **endogenous** because $v_{\text{default}}$ depends on the solved $V_w$ or $W_r$.

## 2.5 Scope: why we stay at fixed prices

To keep the notebook self-contained we **fix** $(r, w, \tau, p)$ as inputs; the Appendix B outer loop (market clearing, SS budget) is not run. Illustrative values line up with Table 7's *baseline with entrepreneurs* row: $r = 6.5\%$, $w \equiv 1$ (Appendix A normalises $\mathbb{E}[y]$ to 1), $p = 0.4 \times w\,\mathbb{E}[y]$, $\tau = 0.10$ (illustrative; the paper does not print a single calibrated $\tau$). See excerpt §12 for the missing outer loop.


---
# Part III — Implementation

## 3.1 Setup and imports


In [1]:
from __future__ import annotations

import sys
from dataclasses import dataclass, replace
from pathlib import Path
from typing import Literal, Tuple

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

# Make Code/Python importable regardless of where the notebook is launched.
_here = Path.cwd().resolve()
REPO_ROOT = _here
if not (REPO_ROOT / 'Code' / 'Python').is_dir():
    for p in [_here, *_here.parents]:
        if (p / 'Code' / 'Python').is_dir():
            REPO_ROOT = p
            break
CODE_PY = REPO_ROOT / 'Code' / 'Python'
if str(CODE_PY) not in sys.path:
    sys.path.insert(0, str(CODE_PY))

from solution import ModelParams, Stage
from stages.cons_noshocks import solve_cons_noshocks, build_a_grid

print('REPO_ROOT:', REPO_ROOT)


REPO_ROOT: /Users/zhengrudan/github/rzheng2112/Cagetti2005tk


## 3.2 Parameter blocks

We separate **three** conceptually different blocks:

| Block | Role | Paper source |
|---|---|---|
| `PaperStructuralParams` | preferences, technology, demographics | Table 5 + text |
| `FixedPriceInputs` | exogenous $(r, w, \tau, \text{pension share})$ | Table 7 / normalisations |
| `SolverSettings` | grids, iteration, damping, EGM stabilisation | numerical (not in paper) |

The `SolverSettings` block is worth explaining:

- `egm_stabilize = 'auto'` — tries raw $\bar v$ first; on failure falls back   to enforcing monotonicity of $\bar v$, then of $c_\delta$. The package's   EGM requires the endogenous cash grid to be increasing; for $\sigma>1$   the transformation `v_to_vinv` also needs $\bar v<0$ samples. We   initialise $V=W=W_r$ at `value_init = -5.0` and cap sampled   continuation at `continuation_value_cap < 0`.
- `damp = 0.58` — convex-combination damping on $(V, W, W_r)$ between   successive Jacobi updates.
- `n_k = 31` — discrete search grid for the entrepreneur's $k$ choice.


In [2]:
# --- A. Paper-side structural parameters (Table 5 + text) ---
@dataclass(frozen=True)
class PaperStructuralParams:
    beta: float = 0.865
    sigma: float = 1.5
    delta: float = 0.06
    nu: float = 0.88
    f: float = 0.75
    pi_y: float = 0.978
    pi_o: float = 0.911
    eta: float = 1.0


paper = PaperStructuralParams()

# Theta grid + transition — Appendix A.
theta_vals = np.array([0.0, 0.514], dtype=float)
P_theta = np.array([[0.964, 0.036], [0.206, 0.794]], dtype=float)

# --- B. Fixed-price inputs ---
# r_net: Table 7 baseline row reports r = 6.5% in the *solved* model.
# w: paper Appendix-A y grid normalised to mean 1; wage is a level normalisation.
# tau: payroll tax in eq. (9); the repo excerpt does not print a single
#      calibrated tau -- 0.10 is illustrative for the inner problem only.
# pension_share: Table 5 rule, p = 40% of average yearly income.
@dataclass(frozen=True)
class FixedPriceInputs:
    r_net: float = 0.065
    w: float = 1.0
    tau: float = 0.10
    pension_share: float = 0.40


fixed_price_inputs = FixedPriceInputs()

# --- C. Numerical solver settings ---
@dataclass(frozen=True)
class SolverSettings:
    use_paper_income_process: bool = True
    a_min: float = 0.0
    a_max: float = 20.0
    a_grid_size: int = 48
    k_max: float = 10.0
    n_k: int = 31
    max_iter: int = 400
    tol: float = 5e-4
    damp: float = 0.58
    egm_stabilize: Literal['auto', 'none', 'v_mono', 'v_and_cdelta'] = 'auto'
    enforce_slack: float = 1e-8
    m_e_floor: float = 1e-10
    theta_zero_value: float = -1e12
    value_init: float = -5.0
    continuation_value_cap: float = -1e-8


solver = SolverSettings()

print('beta:', paper.beta, ' sigma:', paper.sigma, ' f:', paper.f, ' nu:', paper.nu)
print('r_net:', fixed_price_inputs.r_net, ' w:', fixed_price_inputs.w,
      ' tau:', fixed_price_inputs.tau, ' pension_share:', fixed_price_inputs.pension_share)


beta: 0.865  sigma: 1.5  f: 0.75  nu: 0.88
r_net: 0.065  w: 1.0  tau: 0.1  pension_share: 0.4


## 3.3 Income process, fixed-price environment, grids

Appendix A of the paper gives the 5-state Markov chain for $y$. We also keep a 2-state diagonal matrix available as a debugging fallback (toggled by `solver.use_paper_income_process`).

The retiree pension level is constructed as
$$p = \text{pension\_share}\times \underbrace{w \cdot \mathbb{E}[y]}_{\text{reference income}}$$
with $\mathbb{E}[y]$ evaluated under the invariant distribution of $\mathbf{P}_y$. In full GE, *reference income* would be economy-wide average labor income net of taxes.


In [3]:
def paper_appendix_py() -> Tuple[np.ndarray, np.ndarray]:
    y_vals = np.array([0.2468, 0.4473, 0.7654, 1.3097, 2.3742], dtype=float)
    P_y = np.array([
        [0.7376, 0.2473, 0.0150, 0.0002, 0.0000],
        [0.1947, 0.5555, 0.2328, 0.0169, 0.0001],
        [0.0113, 0.2221, 0.5333, 0.2221, 0.0113],
        [0.0001, 0.0169, 0.2328, 0.5555, 0.1947],
        [0.0000, 0.0002, 0.0150, 0.2473, 0.7376],
    ], dtype=float)
    # Appendix print-out is rounded; renormalise rows for floating-point safety.
    P_y = P_y / P_y.sum(axis=1, keepdims=True)
    return y_vals, P_y


def prototype_py() -> Tuple[np.ndarray, np.ndarray]:
    y_vals = np.array([0.4473, 1.3097], dtype=float)
    P_y = np.array([[0.85, 0.15], [0.15, 0.85]], dtype=float)
    return y_vals, P_y


def stationary_dist(P: np.ndarray, tol: float = 1e-12, max_iter: int = 20000) -> np.ndarray:
    n = P.shape[0]
    pi = np.ones(n) / n
    for _ in range(max_iter):
        pi_new = pi @ P
        if np.max(np.abs(pi_new - pi)) < tol:
            return pi_new
        pi = pi_new
    return pi


if solver.use_paper_income_process:
    y_vals, P_y = paper_appendix_py()
else:
    y_vals, P_y = prototype_py()

ny, nth = len(y_vals), len(theta_vals)
pi_y_stat = stationary_dist(P_y)
pi_theta_stat = stationary_dist(P_theta)
E_y = float(pi_y_stat @ y_vals)
reference_income_for_pension = fixed_price_inputs.w * E_y

@dataclass(frozen=True)
class FixedPriceEnv:
    '''Exogenous prices + benefit level for the inner Bellman system.'''
    r: float
    w: float
    tau: float
    p: float
    pension_share: float
    E_y: float
    reference_income_for_pension: float


env = FixedPriceEnv(
    r=fixed_price_inputs.r_net,
    w=fixed_price_inputs.w,
    tau=fixed_price_inputs.tau,
    p=fixed_price_inputs.pension_share * reference_income_for_pension,
    pension_share=fixed_price_inputs.pension_share,
    E_y=E_y,
    reference_income_for_pension=reference_income_for_pension,
)
Rf = 1.0 + env.r

mp_template = ModelParams(
    CRRA=paper.sigma,
    DiscFac=1.0,   # discounting handled manually in continuations
    Rfree=Rf,
    PermGroFac=1.0,
    a_max=solver.a_max,
    a_grid_size=solver.a_grid_size,
)
a_grid = build_a_grid(solver.a_min + 1e-10, solver.a_max, solver.a_grid_size)
na = len(a_grid)
k_candidates = np.linspace(1e-5, solver.k_max, solver.n_k)

print(f'States: y={ny}  theta={nth}  |  grids: a={na}  k={solver.n_k}')
print(f'E[y] under invariant P_y:  {E_y:.6f}')
print(f'Pension level p = {env.p:.6f} = {env.pension_share:.2f} x {reference_income_for_pension:.6f}')
print(f'Gross return R = {Rf:.4f}')


States: y=5  theta=2  |  grids: a=48  k=31
E[y] under invariant P_y:  1.000003
Pension level p = 0.400001 = 0.40 x 1.000003
Gross return R = 1.0650


## 3.4 EGM wrapper, resource equation, expectation operators

Two pieces of machinery sit between the Bellman equations and `solve_cons_noshocks`:

1. **`solve_cons_egm`** — a thin wrapper around `solve_cons_noshocks` that    builds $c_\delta(a)$ from $\bar v(a)$ numerically (since the paper's    mixed continuation is not an analytic object), and tries progressively    stronger monotonicity fixes if the raw $\bar v$ produces a    non-monotone endogenous cash grid.
2. **Expectation operators** — discrete sums over $\mathbf{P}_y$ and    $\mathbf{P}_\theta$ at $a'$ on the asset grid.

For the **entrepreneur** resource equation $m^e = (1-\delta)k + \theta k^\nu - (1+r)(k - a)$ the wrapper is called **once per $(y, \theta)$ slice**; the outer search over $k$ evaluates the EGM stage's interpolated value $v(m^e)$ at each $k$ in `k_candidates`.


In [4]:
def interp_V(V, iy, itheta, ap):
    ap = np.asarray(ap, dtype=float)
    return np.interp(ap, a_grid, V[itheta, iy, :], left=V[itheta, iy, 0], right=V[itheta, iy, -1])


def interp_W(W, itheta, ap):
    ap = np.asarray(ap, dtype=float)
    return np.interp(ap, a_grid, W[itheta, :], left=W[itheta, 0], right=W[itheta, -1])


def interp_Wr(Wr, ap):
    ap = np.asarray(ap, dtype=float)
    return np.interp(ap, a_grid, Wr, left=Wr[0], right=Wr[-1])


def interp_Vw(Vw, iy, itheta, ap):
    ap = np.asarray(ap, dtype=float)
    return np.interp(ap, a_grid, Vw[itheta, iy, :], left=Vw[itheta, iy, 0], right=Vw[itheta, iy, -1])


def EV_young(V, iy, itheta, ap):
    ap = np.asarray(ap, dtype=float)
    s = np.zeros_like(ap, dtype=float)
    for jy in range(ny):
        for jth in range(nth):
            s += P_y[iy, jy] * P_theta[itheta, jth] * interp_V(V, jy, jth, ap)
    return s


def EW_old(W, itheta, ap):
    ap = np.asarray(ap, dtype=float)
    s = np.zeros_like(ap, dtype=float)
    for jth in range(nth):
        s += P_theta[itheta, jth] * interp_W(W, jth, ap)
    return s


def EV_descendant(V, ap, itheta_parent):
    # Descendant: y' ~ invariant(P_y), theta' ~ P_theta(parent).
    ap = np.asarray(ap, dtype=float)
    s = np.zeros_like(ap, dtype=float)
    for jy in range(ny):
        for jth in range(nth):
            s += pi_y_stat[jy] * P_theta[itheta_parent, jth] * interp_V(V, jy, jth, ap)
    return s


def EV_invariant_product(V, ap):
    # Shortcut used in W_r continuation: treats theta' as invariant too.
    ap = np.asarray(ap, dtype=float)
    s = np.zeros_like(ap, dtype=float)
    for jy in range(ny):
        for jth in range(nth):
            s += pi_y_stat[jy] * pi_theta_stat[jth] * interp_V(V, jy, jth, ap)
    return s


def m_entrepreneur(a: float, k: float, theta: float, delta: float, nu: float, Rf: float) -> float:
    return (1.0 - delta) * k + theta * (k ** nu) - Rf * (k - a)


def cδ_from_v_grid(v_vec: np.ndarray, rho: float) -> np.ndarray:
    dv = np.gradient(v_vec, a_grid, edge_order=2)
    dv = np.clip(dv, 1e-14, np.inf)
    return dv ** (-1.0 / rho)


def solve_cons_egm(v_cntn, params: ModelParams, a_min_local: float,
                   mode: Literal['auto', 'none', 'v_mono', 'v_and_cdelta']) -> Tuple[Stage, str]:
    rho = params.CRRA
    v_raw = np.asarray(v_cntn(a_grid), dtype=float)
    v_raw = np.nan_to_num(v_raw, nan=-1e10, posinf=-1e-8, neginf=-1e10)
    v_raw = np.minimum(v_raw, solver.continuation_value_cap)  # guard v_to_vinv

    def attempt(v_vec, cdelta_mono):
        cδ_vec = cδ_from_v_grid(v_vec, rho)
        if cdelta_mono:
            cδ_vec = np.maximum.accumulate(cδ_vec)
        v_fn  = lambda ap, vv=v_vec:  np.interp(np.asarray(ap, float), a_grid, vv)
        cδ_fn = lambda ap, cv=cδ_vec: np.interp(np.asarray(ap, float), a_grid, cv)
        return solve_cons_noshocks(v_fn, cδ_fn, params, a_grid=a_grid, a_min=a_min_local)

    if mode == 'none':
        order = [('none', v_raw, False)]
    elif mode == 'v_mono':
        order = [('v_mono', np.maximum.accumulate(v_raw), False)]
    elif mode == 'v_and_cdelta':
        order = [('v_and_cdelta', np.maximum.accumulate(v_raw), True)]
    else:
        order = [
            ('none', v_raw, False),
            ('v_mono', np.maximum.accumulate(v_raw), False),
            ('v_and_cdelta', np.maximum.accumulate(v_raw), True),
        ]
    last_err = None
    for name, v_vec, cd_mono in order:
        try:
            return attempt(v_vec, cd_mono), name
        except ValueError as e:
            last_err = e
    raise RuntimeError(f'solve_cons_noshocks failed: {last_err}')


## 3.5 Bellman updates and Jacobi value-function iteration

One sweep does: **retiree** ($W_r$, eq. 13) $\to$ **worker** ($V_w$, eq. 8) $\to$ **old entrepreneur** ($W_e$, eq. 11) $\to$ **young entrepreneur** ($V_e$, eq. 3), followed by the max operators $V=\max(V_w, V_e)$, $W=\max(W_e, W_r)$.

Damping is applied on the outer $(V, W, W_r)$ arrays; the inner $V_w, V_e, W_e$ arrays are reported as diagnostic deltas.


In [5]:
def update_Wr(Wr, V):
    def v_cntn(ap):
        ap = np.asarray(ap, dtype=float)
        return (paper.beta * paper.pi_o * interp_Wr(Wr, ap)
                + paper.eta * paper.beta * (1.0 - paper.pi_o) * EV_invariant_product(V, ap))
    stg, _ = solve_cons_egm(v_cntn, mp_template, 0.0, solver.egm_stabilize)
    out = np.zeros_like(Wr)
    for ia, a in enumerate(a_grid):
        out[ia] = stg['dcsn'].v(Rf * a + env.p)
    return out


def update_Vw(V, Wr):
    Vw = np.zeros_like(V)
    for itheta in range(nth):
        for iy in range(ny):
            def v_cntn(ap, iy=iy, itheta=itheta):
                ap = np.asarray(ap, dtype=float)
                return (paper.beta * paper.pi_y * EV_young(V, iy, itheta, ap)
                        + paper.beta * (1.0 - paper.pi_y) * interp_Wr(Wr, ap))
            stg, _ = solve_cons_egm(v_cntn, mp_template, 0.0, solver.egm_stabilize)
            for ia, a in enumerate(a_grid):
                m_w = Rf * a + (1.0 - env.tau) * env.w * y_vals[iy]
                Vw[itheta, iy, ia] = stg['dcsn'].v(m_w)
    return Vw


def update_Ve(V, W, Vw):
    Ve = np.zeros_like(V)
    stats = {'feasible': 0, 'tried': 0, 'enforce_ok': 0, 'egm_modes': []}
    for itheta in range(nth):
        th = float(theta_vals[itheta])
        if th <= 0.0:
            Ve[itheta, :, :] = solver.theta_zero_value
            continue
        for iy in range(ny):
            def v_cntn(ap, iy=iy, itheta=itheta):
                ap = np.asarray(ap, dtype=float)
                return (paper.beta * paper.pi_y * EV_young(V, iy, itheta, ap)
                        + paper.beta * (1.0 - paper.pi_y) * EW_old(W, itheta, ap))
            stg, egm_name = solve_cons_egm(v_cntn, mp_template, 0.0, solver.egm_stabilize)
            stats['egm_modes'].append(('Ve', itheta, iy, egm_name))
            for ia, a in enumerate(a_grid):
                best = float(solver.theta_zero_value)
                for k in k_candidates:
                    stats['tried'] += 1
                    m_e = m_entrepreneur(float(a), float(k), th, paper.delta, paper.nu, Rf)
                    if m_e <= solver.m_e_floor:
                        continue
                    stats['feasible'] += 1
                    v_honor = float(stg['dcsn'].v(m_e))
                    if not np.isfinite(v_honor):
                        continue
                    v_default = interp_Vw(Vw, iy, itheta, paper.f * k)
                    if v_honor >= v_default - solver.enforce_slack:
                        stats['enforce_ok'] += 1
                        if v_honor > best:
                            best = v_honor
                Ve[itheta, iy, ia] = best
    return Ve, stats


def update_We(V, W, Wr):
    We = np.zeros_like(W)
    stats = {'feasible': 0, 'tried': 0, 'enforce_ok': 0, 'egm_modes': []}
    for itheta in range(nth):
        th = float(theta_vals[itheta])
        if th <= 0.0:
            We[itheta, :] = solver.theta_zero_value
            continue
        def v_cntn(ap, itheta=itheta):
            ap = np.asarray(ap, dtype=float)
            return (paper.beta * paper.pi_o * EW_old(W, itheta, ap)
                    + paper.eta * paper.beta * (1.0 - paper.pi_o) * EV_descendant(V, ap, itheta))
        stg, egm_name = solve_cons_egm(v_cntn, mp_template, 0.0, solver.egm_stabilize)
        stats['egm_modes'].append(('We', itheta, egm_name))
        for ia, a in enumerate(a_grid):
            best = float(solver.theta_zero_value)
            for k in k_candidates:
                stats['tried'] += 1
                m_e = m_entrepreneur(float(a), float(k), th, paper.delta, paper.nu, Rf)
                if m_e <= solver.m_e_floor:
                    continue
                stats['feasible'] += 1
                v_honor = float(stg['dcsn'].v(m_e))
                if not np.isfinite(v_honor):
                    continue
                v_default = interp_Wr(Wr, paper.f * k)
                if v_honor >= v_default - solver.enforce_slack:
                    stats['enforce_ok'] += 1
                    if v_honor > best:
                        best = v_honor
            We[itheta, ia] = best
    return We, stats


def jacobi_vfi():
    v0 = float(solver.value_init)
    V  = np.full((nth, ny, na), v0)
    W  = np.full((nth, na), v0)
    Wr = np.full(na, v0)
    Vw_prev = np.full((nth, ny, na), v0)
    Ve_prev = np.full((nth, ny, na), v0)
    We_prev = np.full((nth, na), v0)
    hist, hist_detail = [], []
    om = solver.damp
    st_ve = st_we = None
    for it in range(solver.max_iter):
        V_old, W_old, Wr_old = V.copy(), W.copy(), Wr.copy()
        Wr_new = update_Wr(Wr_old, V_old)
        Vw     = update_Vw(V_old, Wr_new)
        We, st_we = update_We(V_old, W_old, Wr_new)
        Ve, st_ve = update_Ve(V_old, W_old, Vw)

        dVw = float(np.max(np.abs(Vw - Vw_prev)))
        dVe = float(np.max(np.abs(Ve - Ve_prev)))
        dWe = float(np.max(np.abs(We - We_prev)))
        Vw_prev, Ve_prev, We_prev = Vw.copy(), Ve.copy(), We.copy()

        V_new = np.maximum(Vw, Ve)
        W_new = np.maximum(We, Wr_new[None, :])
        dV  = float(np.max(np.abs(V_new - V_old)))
        dW  = float(np.max(np.abs(W_new - W_old)))
        dWr = float(np.max(np.abs(Wr_new - Wr_old)))
        total = dV + dW + dWr
        hist.append(total)
        hist_detail.append(dict(iter=it + 1, dV=dV, dW=dW, dWr=dWr,
                                 dVw=dVw, dVe=dVe, dWe=dWe, sum=total))
        V  = om * V_new  + (1.0 - om) * V_old
        W  = om * W_new  + (1.0 - om) * W_old
        Wr = om * Wr_new + (1.0 - om) * Wr_old
        if total < solver.tol:
            break
    return dict(V=V, W=W, Wr=Wr, Vw=Vw, Ve=Ve, We=We,
                iters=it + 1, hist=np.array(hist), hist_detail=hist_detail,
                stats_ve=st_ve, stats_we=st_we,
                converged=(float(hist[-1]) < solver.tol) if hist else False)


sol = jacobi_vfi()
print(f'Jacobi VFI: converged={sol["converged"]}  iters={sol["iters"]}  '
      f'final sum-norm={sol["hist"][-1]:.2e}')


Jacobi VFI: converged=True  iters=114  final sum-norm=4.95e-04


---
# Part IV — Results

## 4.1 Convergence diagnostics

Reported norms are **sup-norms on the pre-damped update** of each value function. `sum = dV + dW + dWr` is what drives the stopping rule. Inner norms `dVw, dVe, dWe` track each sub-problem between iterations.

The entrepreneur **feasibility rate** is the share of $(a, y, \theta, k)$ quadruples with a positive resource equation $m^e > 0$; the **enforcement-pass** rate is the share of those that also satisfy eq. (5) (young) / eq. (12) (old).


In [6]:
hd = sol['hist_detail']
last = hd[-1]
print('Converged :', sol['converged'], ' iterations :', sol['iters'])
print(f'  last norms: dV={last["dV"]:.2e}  dW={last["dW"]:.2e}  dWr={last["dWr"]:.2e}')
print(f'  inner    : dVw={last["dVw"]:.2e}  dVe={last["dVe"]:.2e}  dWe={last["dWe"]:.2e}')
print(f'  sum (stop criterion): {last["sum"]:.2e}  tol={solver.tol:.1e}')

ve, we = sol['stats_ve'], sol['stats_we']
print()
print(f'Young entrepreneur: feasible {ve["feasible"]}/{ve["tried"]}  '
      f'enforce_ok {ve["enforce_ok"]}  '
      f'(enforce pass rate {ve["enforce_ok"]/max(1, ve["feasible"]):.3f})')
print(f'Old entrepreneur  : feasible {we["feasible"]}/{we["tried"]}  '
      f'enforce_ok {we["enforce_ok"]}  '
      f'(enforce pass rate {we["enforce_ok"]/max(1, we["feasible"]):.3f})')

try:
    from IPython.display import display
except ImportError:
    display = print

fig_c, ax_c = plt.subplots(1, 2, figsize=(11, 4))
h = sol['hist']
ax_c[0].semilogy(np.arange(1, len(h) + 1), h, marker='.')
ax_c[0].set_xlabel('iteration'); ax_c[0].set_ylabel('dV + dW + dWr')
ax_c[0].set_title('Total change (log)')
dd = sol['hist_detail']
ax_c[1].semilogy([d['iter'] for d in dd], [d['dV'] for d in dd], label='dV',  marker='.')
ax_c[1].semilogy([d['iter'] for d in dd], [d['dW'] for d in dd], label='dW',  marker='.')
ax_c[1].semilogy([d['iter'] for d in dd], [d['dWr'] for d in dd], label='dWr', marker='.')
ax_c[1].set_xlabel('iteration'); ax_c[1].legend()
ax_c[1].set_title('Per-function update norms')
plt.tight_layout()
display(fig_c)


Converged : True  iterations : 114
  last norms: dV=1.63e-04  dW=1.66e-04  dWr=1.66e-04
  inner    : dVw=8.81e-05  dVe=8.86e-05  dWe=8.97e-05
  sum (stop criterion): 4.95e-04  tol=5.0e-04

Young entrepreneur: feasible 7440/7440  enforce_ok 577  (enforce pass rate 0.078)
Old entrepreneur  : feasible 1488/1488  enforce_ok 1012  (enforce pass rate 0.680)


<Figure size 1100x400 with 2 Axes>

## 4.2 Value functions, policies, and the entrepreneur $k^\star$

The first panel row compares the **occupational** envelope at a middle productivity slice. Where the entrepreneur value (orange) crosses above the worker value (blue), the young optimal choice flips from worker to entrepreneur.

The second row shows implied **policies**: worker $a'(a)$ under low and high $y$, and the retiree's $a'(a)$.

The third row shows the **discrete** $k^\star$ (argmax over `k_candidates`) for young and old entrepreneurs at $\theta_{\text{high}}$. Since the Bellman already restricts to enforcement-feasible $k$, the winning $k^\star$ automatically passes eq. (5)/(12).


In [7]:
def implied_c_ap_worker(iy, itheta, V, Wr):
    def v_cntn(ap):
        ap = np.asarray(ap, dtype=float)
        return (paper.beta * paper.pi_y * EV_young(V, iy, itheta, ap)
                + paper.beta * (1.0 - paper.pi_y) * interp_Wr(Wr, ap))
    stg, _ = solve_cons_egm(v_cntn, mp_template, 0.0, solver.egm_stabilize)
    c_arr, ap_arr = np.zeros(na), np.zeros(na)
    for ia, a in enumerate(a_grid):
        m_w = Rf * a + (1.0 - env.tau) * env.w * y_vals[iy]
        c_arr[ia] = float(stg['dcsn'].c(m_w))
        ap_arr[ia] = m_w - c_arr[ia]
    return c_arr, ap_arr


def implied_c_ap_retiree(V, Wr):
    def v_cntn(ap):
        ap = np.asarray(ap, dtype=float)
        return (paper.beta * paper.pi_o * interp_Wr(Wr, ap)
                + paper.eta * paper.beta * (1.0 - paper.pi_o) * EV_invariant_product(V, ap))
    stg, _ = solve_cons_egm(v_cntn, mp_template, 0.0, solver.egm_stabilize)
    c_arr, ap_arr = np.zeros(na), np.zeros(na)
    for ia, a in enumerate(a_grid):
        m_r = Rf * a + env.p
        c_arr[ia] = float(stg['dcsn'].c(m_r))
        ap_arr[ia] = m_r - c_arr[ia]
    return c_arr, ap_arr


def k_star_young(V, W, Wr, Vw):
    k_star = np.full((nth, ny, na), np.nan)
    for itheta in range(nth):
        th = float(theta_vals[itheta])
        if th <= 0.0:
            continue
        for iy in range(ny):
            def v_cntn(ap, iy=iy, itheta=itheta):
                ap = np.asarray(ap, dtype=float)
                return (paper.beta * paper.pi_y * EV_young(V, iy, itheta, ap)
                        + paper.beta * (1.0 - paper.pi_y) * EW_old(W, itheta, ap))
            stg, _ = solve_cons_egm(v_cntn, mp_template, 0.0, solver.egm_stabilize)
            for ia, a in enumerate(a_grid):
                best = float(solver.theta_zero_value); best_k = np.nan
                for k in k_candidates:
                    m_e = m_entrepreneur(float(a), float(k), th, paper.delta, paper.nu, Rf)
                    if m_e <= solver.m_e_floor:
                        continue
                    v_honor = float(stg['dcsn'].v(m_e))
                    if not np.isfinite(v_honor):
                        continue
                    v_default = interp_Vw(Vw, iy, itheta, paper.f * k)
                    if v_honor >= v_default - solver.enforce_slack and v_honor > best:
                        best, best_k = v_honor, k
                k_star[itheta, iy, ia] = best_k
    return k_star


def k_star_old(V, W, Wr):
    k_star = np.full((nth, na), np.nan)
    for itheta in range(nth):
        th = float(theta_vals[itheta])
        if th <= 0.0:
            continue
        def v_cntn(ap, itheta=itheta):
            ap = np.asarray(ap, dtype=float)
            return (paper.beta * paper.pi_o * EW_old(W, itheta, ap)
                    + paper.eta * paper.beta * (1.0 - paper.pi_o) * EV_descendant(V, ap, itheta))
        stg, _ = solve_cons_egm(v_cntn, mp_template, 0.0, solver.egm_stabilize)
        for ia, a in enumerate(a_grid):
            best = float(solver.theta_zero_value); best_k = np.nan
            for k in k_candidates:
                m_e = m_entrepreneur(float(a), float(k), th, paper.delta, paper.nu, Rf)
                if m_e <= solver.m_e_floor:
                    continue
                v_honor = float(stg['dcsn'].v(m_e))
                if not np.isfinite(v_honor):
                    continue
                v_default = interp_Wr(Wr, paper.f * k)
                if v_honor >= v_default - solver.enforce_slack and v_honor > best:
                    best, best_k = v_honor, k
            k_star[itheta, ia] = best_k
    return k_star


V_, W_, Wr_, Vw_, Ve_, We_ = (sol['V'], sol['W'], sol['Wr'],
                               sol['Vw'], sol['Ve'], sol['We'])
young_ent = Ve_ >= Vw_
old_ent   = We_ >= Wr_[None, :]
k_y = k_star_young(V_, W_, Wr_, Vw_)
k_o = k_star_old(V_, W_, Wr_)

iy_mid, iy_lo, iy_hi = ny // 2, 0, ny - 1
ith_hi = min(1, nth - 1)
c_w_lo, ap_w_lo = implied_c_ap_worker(iy_lo, ith_hi, V_, Wr_)
c_w_hi, ap_w_hi = implied_c_ap_worker(iy_hi, ith_hi, V_, Wr_)
c_r,    ap_r    = implied_c_ap_retiree(V_, Wr_)

print('=== Young occupational branch (V_e >= V_w, share of asset grid) ===')
for it in range(nth):
    for iy in range(ny):
        print(f'  theta[{it}]={theta_vals[it]:.3f}  y[{iy}]={y_vals[iy]:.3f}  '
              f'P(V_e>=V_w) = {float(np.mean(young_ent[it, iy, :])):.3f}')
print('=== Old branch (W_e >= W_r) ===')
for it in range(nth):
    print(f'  theta[{it}]={theta_vals[it]:.3f}  P(W_e>=W_r) = {float(np.mean(old_ent[it, :])):.3f}')

fig, ax = plt.subplots(3, 2, figsize=(11, 11))
# Row 1: value functions at (theta_high, y_mid)
ax[0, 0].plot(a_grid, Vw_[ith_hi, iy_mid, :], label='$V_w$')
ax[0, 0].plot(a_grid, Ve_[ith_hi, iy_mid, :], label='$V_e$')
ax[0, 0].plot(a_grid, V_ [ith_hi, iy_mid, :], label='$V$', linestyle='--')
ax[0, 0].legend(); ax[0, 0].set_title('Young values (theta high, y mid)'); ax[0, 0].set_xlabel('a')
ax[0, 1].plot(a_grid, We_[ith_hi, :], label='$W_e$')
ax[0, 1].plot(a_grid, Wr_,            label='$W_r$')
ax[0, 1].plot(a_grid, W_ [ith_hi, :], label='$W$', linestyle='--')
ax[0, 1].legend(); ax[0, 1].set_title('Old values (theta high)'); ax[0, 1].set_xlabel('a')
# Row 2: policies
ax[1, 0].plot(a_grid, ap_w_lo, label=f"worker a'  y={y_vals[iy_lo]:.2f}")
ax[1, 0].plot(a_grid, ap_w_hi, label=f"worker a'  y={y_vals[iy_hi]:.2f}")
ax[1, 0].plot(a_grid, a_grid, 'k:', alpha=0.4, label='45 deg')
ax[1, 0].legend(); ax[1, 0].set_xlabel('a'); ax[1, 0].set_title("Worker next-period assets (theta high)")
ax[1, 1].plot(a_grid, ap_r, color='C2')
ax[1, 1].plot(a_grid, a_grid, 'k:', alpha=0.4)
ax[1, 1].set_xlabel('a'); ax[1, 1].set_title("Retiree next-period assets")
# Row 3: k_star
ax[2, 0].plot(a_grid, k_y[ith_hi, iy_mid, :], label='young $k^*$ (theta high, y mid)')
ax[2, 0].plot(a_grid, k_o[ith_hi, :],         label='old $k^*$ (theta high)')
ax[2, 0].set_xlabel('a'); ax[2, 0].legend(); ax[2, 0].set_title('Entrepreneur $k^*$ (discrete argmax)')
im = ax[2, 1].imshow(young_ent[ith_hi, :, :].astype(float), aspect='auto',
                     origin='lower', extent=[a_grid[0], a_grid[-1], -0.5, ny - 0.5])
ax[2, 1].set_xlabel('a'); ax[2, 1].set_ylabel('y index')
ax[2, 1].set_title('Young: $V_e \\geq V_w$ (theta high)')
plt.tight_layout()
display(fig)


=== Young occupational branch (V_e >= V_w, share of asset grid) ===
  theta[0]=0.000  y[0]=0.247  P(V_e>=V_w) = 0.000
  theta[0]=0.000  y[1]=0.447  P(V_e>=V_w) = 0.000
  theta[0]=0.000  y[2]=0.765  P(V_e>=V_w) = 0.000
  theta[0]=0.000  y[3]=1.310  P(V_e>=V_w) = 0.000
  theta[0]=0.000  y[4]=2.374  P(V_e>=V_w) = 0.000
  theta[1]=0.514  y[0]=0.247  P(V_e>=V_w) = 0.250
  theta[1]=0.514  y[1]=0.447  P(V_e>=V_w) = 0.146
  theta[1]=0.514  y[2]=0.765  P(V_e>=V_w) = 0.104
  theta[1]=0.514  y[3]=1.310  P(V_e>=V_w) = 0.062
  theta[1]=0.514  y[4]=2.374  P(V_e>=V_w) = 0.042
=== Old branch (W_e >= W_r) ===
  theta[0]=0.000  P(W_e>=W_r) = 0.000
  theta[1]=0.514  P(W_e>=W_r) = 1.000


<Figure size 1100x1100 with 6 Axes>

## 4.3 Comparative statics (partial-equilibrium)

Each row re-solves the inner Bellman after changing **one** input at a time, then restores the baseline. Only directional interpretation is valid — coarse grids and fixed prices remain.


In [8]:
def _metrics(ss):
    iy_m = ny // 2; ith = min(1, nth - 1)
    br_y = float(np.mean(ss['Ve'][ith, iy_m, :] >= ss['Vw'][ith, iy_m, :]))
    br_o = float(np.mean(ss['We'][ith, :] >= ss['Wr']))
    return dict(converged=ss['converged'], iters=ss['iters'],
                branch_young_mid=br_y, branch_old_high=br_o,
                sum_norm=float(ss['hist'][-1]) if len(ss['hist']) else float('nan'))


_bk = dict(env=env, Rf=Rf, mp_template=mp_template, paper=paper,
           fixed_price_inputs=fixed_price_inputs, theta_vals=theta_vals.copy(),
           k_candidates=k_candidates.copy(), sol=sol, solver=solver)
E_y_ref = float(pi_y_stat @ y_vals)


def _rebuild_env(fpi):
    ref = fpi.w * E_y_ref
    return FixedPriceEnv(r=fpi.r_net, w=fpi.w, tau=fpi.tau,
                         p=fpi.pension_share * ref,
                         pension_share=fpi.pension_share,
                         E_y=E_y_ref, reference_income_for_pension=ref)


def _restore():
    global env, Rf, mp_template, paper, fixed_price_inputs, theta_vals, k_candidates, sol, solver
    env = _bk['env']; Rf = _bk['Rf']; mp_template = _bk['mp_template']
    paper = _bk['paper']; fixed_price_inputs = _bk['fixed_price_inputs']
    theta_vals = _bk['theta_vals'].copy(); k_candidates = _bk['k_candidates'].copy()
    sol = _bk['sol']; solver = _bk['solver']


def _run(label, fpi=None, paper_p=None, thv=None, kc=None):
    global env, Rf, mp_template, paper, fixed_price_inputs, theta_vals, k_candidates, sol, solver
    _restore()
    solver = replace(solver, max_iter=500)
    if fpi is not None:
        fixed_price_inputs = fpi
        env = _rebuild_env(fpi); Rf = 1.0 + env.r
        mp_template = ModelParams(CRRA=paper.sigma, DiscFac=1.0, Rfree=Rf,
                                   PermGroFac=1.0, a_max=solver.a_max,
                                   a_grid_size=solver.a_grid_size)
    if paper_p is not None:
        paper = paper_p
    if thv is not None:
        theta_vals = thv
    if kc is not None:
        k_candidates = kc
    sol = jacobi_vfi()
    m = _metrics(sol); _restore()
    return label, m


rows = [('baseline', _metrics(sol))]
rows.append(_run('r = 7.5%',            fpi=replace(fixed_price_inputs, r_net=0.075)))
rows.append(_run('tau = 0.12',          fpi=replace(fixed_price_inputs, tau=0.12)))
rows.append(_run('pension_share=0.38',  fpi=replace(fixed_price_inputs, pension_share=0.38)))
rows.append(_run('f = 0.78 (tighter)',  paper_p=replace(paper, f=0.78)))
rows.append(_run('theta_hi = 0.55',     thv=np.array([0.0, 0.55], dtype=float)))
rows.append(_run('n_k = 35 (denser)',   kc=np.linspace(1e-5, solver.k_max, 35)))

print(f'{"scenario":<22}  {"iters":>5}  {"branch_young_mid":>17}  {"branch_old_high":>16}  sum_norm')
for name, m in rows:
    print(f'{name:<22}  {m["iters"]:>5}  {m["branch_young_mid"]:>17.4f}  '
          f'{m["branch_old_high"]:>16.4f}  {m["sum_norm"]:.2e}')


scenario                iters   branch_young_mid   branch_old_high  sum_norm
baseline                  114             0.1042            1.0000  4.95e-04
r = 7.5%                  115             0.1042            1.0000  4.71e-04
tau = 0.12                115             0.1042            1.0000  4.77e-04
pension_share=0.38        115             0.1042            1.0000  4.83e-04
f = 0.78 (tighter)        115             0.1042            1.0000  4.82e-04
theta_hi = 0.55           115             0.1042            1.0000  4.74e-04
n_k = 35 (denser)         115             0.1042            1.0000  4.78e-04


## 4.4 Stationary distribution — young-only kernel approximation

For a **quick** check that the model's entrepreneur share is in the right ballpark, we iterate a young-only transition kernel with $\pi_y = 1$ (young stay young; no aging, no descendants). Policies come from the converged Bellman; next-period assets are linearly histogrammed onto the existing asset grid.

This is **not** the paper's full demographic stationary measure — see Section 11 — but it is a meaningful diagnostic: the mass on the entrepreneur region $\{V_e \ge V_w\}$ times the within-$\theta$ transitions gives a first-order estimate of the entrepreneur fraction.


In [9]:
def build_ap_young_full(V, W, Wr, Vw, Ve):
    ap_w = np.zeros((nth, ny, na))
    ap_e = np.zeros((nth, ny, na))
    occ = Ve >= Vw
    for itheta in range(nth):
        for iy in range(ny):
            _, ap_w[itheta, iy, :] = implied_c_ap_worker(iy, itheta, V, Wr)
            th = float(theta_vals[itheta])
            if th <= 0.0:
                ap_e[itheta, iy, :] = np.nan; continue
            def v_cntn(ap, iy=iy, itheta=itheta):
                ap = np.asarray(ap, dtype=float)
                return (paper.beta * paper.pi_y * EV_young(V, iy, itheta, ap)
                        + paper.beta * (1.0 - paper.pi_y) * EW_old(W, itheta, ap))
            stg, _ = solve_cons_egm(v_cntn, mp_template, 0.0, solver.egm_stabilize)
            for ia, a in enumerate(a_grid):
                best = float(solver.theta_zero_value); best_ap = np.nan
                for k in k_candidates:
                    m_e = m_entrepreneur(float(a), float(k), th, paper.delta, paper.nu, Rf)
                    if m_e <= solver.m_e_floor:
                        continue
                    v_honor = float(stg['dcsn'].v(m_e))
                    if not np.isfinite(v_honor):
                        continue
                    v_default = interp_Vw(Vw, iy, itheta, paper.f * k)
                    if v_honor >= v_default - solver.enforce_slack and v_honor > best:
                        best = v_honor
                        best_ap = m_e - float(stg['dcsn'].c(m_e))
                ap_e[itheta, iy, ia] = best_ap
    return ap_w, ap_e, occ


def _hist_add2(mu, jθ, jy, ap, mass):
    ap = float(np.clip(ap, a_grid[0], a_grid[-1]))
    idx = int(np.searchsorted(a_grid, ap) - 1)
    idx = int(np.clip(idx, 0, na - 2))
    t = (ap - a_grid[idx]) / (a_grid[idx + 1] - a_grid[idx] + 1e-18)
    mu[jθ, jy, idx]     += mass * (1.0 - t)
    mu[jθ, jy, idx + 1] += mass * t


def iterate_mu_young_kernel(mu, ap_w, ap_e, occ, max_iter=8000, tol=1e-9):
    hist = []
    mu = np.asarray(mu, dtype=float); mu /= np.sum(mu)
    for _ in range(max_iter):
        mu_new = np.zeros_like(mu)
        for iθ in range(nth):
            for iy in range(ny):
                for ia in range(na):
                    m = mu[iθ, iy, ia]
                    if m <= 0.0:
                        continue
                    ap = ap_e[iθ, iy, ia] if occ[iθ, iy, ia] else ap_w[iθ, iy, ia]
                    if occ[iθ, iy, ia] and not np.isfinite(ap):
                        ap = ap_w[iθ, iy, ia]
                    for jθ in range(nth):
                        for jy in range(ny):
                            _hist_add2(mu_new, jθ, jy, ap,
                                       m * P_theta[iθ, jθ] * P_y[iy, jy])
        s = np.sum(mu_new);
        if s > 0: mu_new /= s
        d = float(np.max(np.abs(mu_new - mu))); hist.append(d)
        mu = mu_new
        if d < tol:
            break
    return mu, hist


ap_w_f, ap_e_f, occ_f = build_ap_young_full(V_, W_, Wr_, Vw_, Ve_)
mu0 = np.ones((nth, ny, na), dtype=float); mu0 /= np.sum(mu0)
mu_star, mu_hist = iterate_mu_young_kernel(mu0, ap_w_f, ap_e_f, occ_f)

ent_share_mu   = float(np.sum(mu_star * occ_f))
mean_a_mu      = float(np.sum(mu_star * a_grid.reshape(1, 1, -1)))
w_mass         = float(np.sum(mu_star * (~occ_f)))
avg_worker_lab = float(np.sum(mu_star * (~occ_f) * (1.0 - env.tau) * env.w
                              * y_vals.reshape(1, ny, 1))) / max(w_mass, 1e-14)
avg_y_mu       = float(np.sum(mu_star * y_vals.reshape(1, ny, 1)))

print(f'mu iterations                 : {len(mu_hist)}   last step: {mu_hist[-1]:.2e}')
print(f'entrepreneur share under mu   : {ent_share_mu:.4f}   '
      f'(paper Table 6 baseline: 0.0750)')
print(f'mean assets <a>               : {mean_a_mu:.4f}')
print(f'E[y] under mu (productivity)  : {avg_y_mu:.4f}')
print(f'avg (1-tau) w y  | workers    : {avg_worker_lab:.4f}')
print(f'env.reference_income_for_pension: {env.reference_income_for_pension:.4f}')

fig_mu, ax_mu = plt.subplots(1, 2, figsize=(11, 4))
ma = np.sum(mu_star, axis=(0, 1))
ax_mu[0].bar(np.arange(na), ma)
ax_mu[0].set_xlabel('asset index'); ax_mu[0].set_title('Marginal $\\mu$ over assets')
ax_mu[1].semilogy(np.arange(1, len(mu_hist) + 1), mu_hist, marker='.')
ax_mu[1].set_xlabel('iteration'); ax_mu[1].set_title('$\\mu$ kernel iteration diff')
plt.tight_layout()
display(fig_mu)


mu iterations                 : 173   last step: 9.39e-10
entrepreneur share under mu   : 0.0731   (paper Table 6 baseline: 0.0750)
mean assets <a>               : 2.2908
E[y] under mu (productivity)  : 1.0000
avg (1-tau) w y  | workers    : 0.9028
env.reference_income_for_pension: 1.0000


<Figure size 1100x400 with 2 Axes>

---
# Part V — Summary, scope, next steps

## What we reproduced

- The Bellman system of eqs. (2)–(14) with the paper's **calibrated**   parameters (Table 5 panel B) and **fixed** structural parameters   (Table 5 panel A).
- The paper's **Appendix A** 5-state labor process and 2-state $\theta$   process.
- The **endogenous borrowing limit** implicit in eqs. (5) and (12), via   the Bellman-level reformulation used in the current literature: at each   grid point we restrict the $\max$ to $k$ satisfying   $v_{\text{honour}} \ge v_{\text{default}}$.
- Qualitative patterns expected from the paper: old entrepreneurs   overwhelmingly continue (high $W_e \ge W_r$ share); young   entrepreneurship is rarer and concentrated at low-$y$, high-$\theta$   workers where the return from labor is low relative to the outside   option.
- A rough **entrepreneur-share** estimate from the young-only kernel   $\mu$ ≈ **7.3%**, within 0.3 percentage points of the paper's baseline   (7.5%).

## What is still needed for *full* replication

| Component | Status here | What it requires |
|---|---|---|
| GE prices $(r, w)$ | fixed at Table-7 baseline | outer loop updating $(r, w)$ from the corporate FOC $Y_c = A K_c^\alpha L_c^{1-\alpha}$ and capital-market clearing |
| SS budget $(\tau, p)$ | $\tau$ illustrative; $p$ from rule with $\mathbb{E}[y]$ | budget-balance fixed point: $\tau w L = p \cdot N_{\text{retirees}}$ |
| Full demographic $\mu$ | young-only kernel w/ $\pi_y = 1$ | flows young$\to$old, old$\to$descendant, correct $\theta'$ conditioning at birth |
| Table 6 moments | only entrepreneur share approximated | $\mu$-based wealth CDF $\Rightarrow$ Gini, top $p$%, K/Y |
| Table 7 counterfactuals | comparative statics only | re-calibrate under $f=0.85$, $\eta=0$, etc. |
| Table 5 calibration | parameters taken as given | SMM / moment-matching on (β, θ$_+$, $\mathbf{P}_\theta$, $\nu$, $f$) |

## Suggested next steps (in order)

1. Replace the young-only kernel with a full three-type stationary    distribution (YW + YE together, OE, OR) respecting $\pi_y$, $\pi_o$, and    descendant birth.
2. Wrap Jacobi VFI in a capital-market-clearing outer loop over $(r, K_c)$    with a corporate FOC; enforce SS balance simultaneously.
3. Compute the wealth CDF, top shares, and K/Y from $\mu$; compare to    Table 6.
4. Redo the baseline with $f=0.85$, $\eta=0$, $(\eta=0, \beta=0.88)$ for    Table 7 counterfactuals.
5. Introduce SMM over the five calibrated parameters; optimise against the    weighted Table 6 moment vector.

## References

- Cagetti, M. & De Nardi, M. (2006). *Entrepreneurship, Frictions, and   Wealth.* Journal of Political Economy, 114(5): 835–870.
- Carroll, C. (2006). *The method of endogenous gridpoints for solving   dynamic stochastic optimization problems.* Economics Letters.
- SolvingMicroDSOPs: lecture notes + Python/Stata replication code   (this repository).
